# 07. Phân loại (E1 - E8)

Notebook này xây dựng các mô hình học máy để dự đoán xem một khối code AI có bị chỉnh sửa trong 30 ngày đầu hay không (`change_30d`). Tập huấn luyện và kiểm tra được chia theo từng `repo` để tránh rò rỉ dữ liệu giữa các khối code cùng dự án.

Import thư viện, tải dữ liệu và lọc các khối code đủ điều kiện theo dõi 30 ngày.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/processed/commits_features.csv")
REPORT_DIR = Path("../reports")
FIG_DIR = REPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False).replace([np.inf, -np.inf], np.nan)
df = df[df["change_30d"].notna()].copy()
df["change_30d"] = df["change_30d"].astype(int)
df["log_code_lines"] = np.log1p(df["code_lines"])
df["has_function_def"] = df["has_function_def"].astype(int)
df["has_class_def"] = df["has_class_def"].astype(int)

numeric_features = ["log_code_lines", "comment_chars", "has_function_def", "has_class_def"]
categorical_features = ["language", "ai_tool", "ai_verb", "file_role"]
feature_cols = numeric_features + categorical_features

df = df.dropna(subset=feature_cols + ["change_30d", "repo"]).copy()
X = df[feature_cols]
y = df["change_30d"]
groups = df["repo"]

print(f"Eligible dataset: {len(df):,} blocks, {groups.nunique():,} repos, positive rate: {y.mean():.3f}")

Eligible dataset: 19,554 blocks, 11,234 repos, positive rate: 0.230


Chia tập train/test (80/20) theo nhóm `repo` và thiết lập pipeline tiền xử lý cùng 3 mô hình: Logistic Regression, Decision Tree và Random Forest.

In [2]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_leaf=15, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, max_depth=14, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=42),
}

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train):,} blocks; Test: {len(X_test):,} blocks ({groups.iloc[test_idx].nunique():,} repos)")

Train: 14,262 blocks; Test: 5,292 blocks (2,247 repos)


## E1 - E4. Huấn luyện và đánh giá trên tập Test

Huấn luyện lần lượt 3 mô hình trên tập train và in báo cáo phân loại trên tập test.

In [3]:
pipelines = {
    name: Pipeline([("preprocess", preprocessor), ("model", model)])
    for name, model in models.items()
}

fitted = {}
predictions = {}
probabilities = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    predictions[name] = pipe.predict(X_test)
    probabilities[name] = pipe.predict_proba(X_test)[:, 1]
    print(f"--- {name} ---")
    print(classification_report(y_test, predictions[name], target_names=["No 30d change", "Changed in 30d"]))

--- Logistic Regression ---
                precision    recall  f1-score   support

 No 30d change       0.80      0.65      0.71      3992
Changed in 30d       0.31      0.50      0.38      1300

      accuracy                           0.61      5292
     macro avg       0.55      0.57      0.55      5292
  weighted avg       0.68      0.61      0.63      5292

--- Decision Tree ---
                precision    recall  f1-score   support

 No 30d change       0.80      0.43      0.56      3992
Changed in 30d       0.28      0.67      0.39      1300

      accuracy                           0.49      5292
     macro avg       0.54      0.55      0.47      5292
  weighted avg       0.67      0.49      0.52      5292

--- Random Forest ---
                precision    recall  f1-score   support

 No 30d change       0.80      0.67      0.73      3992
Changed in 30d       0.32      0.47      0.38      1300

      accuracy                           0.62      5292
     macro avg       0.5

## E5. Tổng hợp chỉ số đánh giá, ma trận nhầm lẫn và đường cong ROC

Lưu bảng chỉ số Accuracy, Precision, Recall, F1, ROC-AUC và vẽ hình minh họa.

In [4]:
metric_rows = []
for name in models:
    pred, prob = predictions[name], probabilities[name]
    metric_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob),
    })

metrics = pd.DataFrame(metric_rows).sort_values("roc_auc", ascending=False)
display(metrics)
metrics.to_csv(REPORT_DIR / "classification_metrics.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["No 30d change", "Changed in 30d"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
fig.tight_layout()
fig.savefig(FIG_DIR / "E5_confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
for name, prob in probabilities.items():
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
ax.set_title("ROC curves (change_30d)")
fig.tight_layout()
fig.savefig(FIG_DIR / "E5_roc_curves.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

,model,accuracy,precision,recall,f1,roc_auc
2,Random Forest,0.622638,0.318016,0.468462,0.378849,0.600797
0,Logistic Regression,0.608655,0.312773,0.495385,0.383447,0.600020
1,Decision Tree,0.487339,0.276636,0.673077,0.392113,0.560995


## E6. Mức độ quan trọng của các đặc trưng (Feature Importance)

Xem những đặc trưng nào đóng góp nhiều nhất vào dự đoán của mô hình.

In [5]:
importance_rows = []
for name, pipe in fitted.items():
    model = pipe.named_steps["model"]
    prep = pipe.named_steps["preprocess"]
    names = prep.get_feature_names_out()
    values = model.coef_[0] if hasattr(model, "coef_") else model.feature_importances_
    for feature, value in zip(names, values):
        importance_rows.append({
            "model": name,
            "feature": feature,
            "importance": value,
            "abs_importance": abs(value),
        })

importance = pd.DataFrame(importance_rows)
display(importance.sort_values(["model", "abs_importance"], ascending=[True, False]).groupby("model").head(12))
importance.to_csv(REPORT_DIR / "classification_feature_importance.csv", index=False)

rf_top = importance[importance["model"] == "Random Forest"].nlargest(15, "abs_importance").sort_values("importance")
plt.figure(figsize=(9, 5.5))
sns.barplot(data=rf_top, x="importance", y="feature", color="#16a34a")
plt.title("Top 15 Random Forest feature importance (change_30d)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(FIG_DIR / "E6_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()

,model,feature,importance,abs_importance
20,Decision Tree,num__log_code_lines,0.454785,0.454785
21,Decision Tree,num__comment_chars,0.284572,0.284572
37,Decision Tree,cat__file_role_frontend_web,0.038708,0.038708
26,Decision Tree,cat__ai_tool_chatgpt_gpt,0.028912,0.028912
22,Decision Tree,num__has_function_def,0.026297,0.026297
25,Decision Tree,cat__language_python,0.025375,0.025375
29,Decision Tree,cat__ai_tool_github_copilot,0.023551,0.023551
31,Decision Tree,cat__ai_tool_other_ai,0.016867,0.016867
24,Decision Tree,cat__language_js,0.016739,0.016739
35,Decision Tree,cat__file_role_core_source,0.015046,0.015046


## E7. Kiểm chứng chéo 5-fold GroupKFold theo repo

Chạy đánh giá chéo 5 lần chia theo từng repo để kiểm tra độ ổn định của các mô hình.

In [6]:
cv = GroupKFold(n_splits=5)
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}
cv_rows = []

for name, pipe in pipelines.items():
    result = cross_validate(pipe, X, y, groups=groups, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"model": name}
    for metric in scoring:
        row[f"{metric}_mean"] = result[f"test_{metric}"].mean()
        row[f"{metric}_std"] = result[f"test_{metric}"].std()
    cv_rows.append(row)

cv_table = pd.DataFrame(cv_rows).sort_values("roc_auc_mean", ascending=False)
display(cv_table)
cv_table.to_csv(REPORT_DIR / "classification_cross_validation.csv", index=False)

,model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
2,Random Forest,0.602793,0.007809,0.290212,0.014972,0.503402,0.024226,0.367560,0.010195,0.594637,0.006378
0,Logistic Regression,0.573591,0.011561,0.273927,0.015026,0.518453,0.020173,0.358169,0.014548,0.580790,0.008177
1,Decision Tree,0.543214,0.012269,0.265477,0.012211,0.558235,0.036936,0.358889,0.005710,0.563811,0.007576


## E8. Thử nghiệm dự đoán mục đích sửa lỗi (`is_bug_fix`)

Trên các khối code có ghi nhận thay đổi, kiểm tra xem có thể dự đoán lần sửa đầu tiên là sửa lỗi (`bug_fix`) hay không.

In [7]:
changed = df[df["has_recorded_change"] == True].copy()
if len(set(changed["change_intent"].unique()) - {"other_change", "no_recorded_change"}) == 0:
    print("Bo qua E8 vi chua co cot change_commit_message tu notebook 01.")
else:
    changed["is_bug_fix"] = (changed["change_intent"] == "bug_fix").astype(int)
    X_bug = changed[feature_cols]
    y_bug = changed["is_bug_fix"]
    g_bug = changed["repo"]
    res_bug = cross_validate(pipelines["Random Forest"], X_bug, y_bug, groups=g_bug, cv=cv, scoring=["f1", "roc_auc"], n_jobs=-1)
    print(f"Bug-fix 5-fold GroupKFold ROC-AUC: {res_bug['test_roc_auc'].mean():.3f} +/- {res_bug['test_roc_auc'].std():.3f}")

Bug-fix 5-fold GroupKFold ROC-AUC: 0.594 +/- 0.022
